# Customer Churn Feature Pipeline
This notebook constructs leakage-safe predictive features for the churn modeling dataset.

Each row represents one eligible customer at one snapshot date.

All predictor variables are constructed exclusively from information available on or before the corresponding snapshot date. Future activity and churn-construction variables are never used as model features.

## 1. Pipeline Configuration

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd


PROJECT_ROOT = Path(
    "/Users/qiaodou/Desktop/onemoreclass-churn-intelligence"
)

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.activity import build_activity_features
from src.features.transactions import build_transaction_features
from src.features.interactions import build_interaction_features
from src.features.validation import validate_feature_matrix


RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

SNAPSHOT_KEY = ["customer_id", "snapshot_date"]
TARGET_COLUMN = "churn"
ACTIVITY_WINDOWS = [7, 14, 30, 60]

In [2]:
FORBIDDEN_FEATURES = {
    "future_active_days",
    "future_logins",
    "future_minutes",
    "future_assignments",
    "future_active_day_rate",
    "future_minutes_per_day",
    "future_assignments_per_day",
    "future_login_decline",
    "future_minutes_decline",
    "future_assignment_decline",
    "login_decline_flag",
    "minutes_decline_flag",
    "assignment_decline_flag",
    "decline_signal_count",
}


EXPECTED_NULLABLE_FEATURES = {
    # Transaction history
    "transaction_count",
    "paid_transactions",
    "failed_transactions",
    "pending_transactions",
    "avg_discount_rate",
    "discount_purchase_share",
    "live_purchase_share",
    "days_since_last_purchase",
    "payment_success_rate",
    "failed_payment_rate",
    "pending_payment_rate",

    # Interaction history
    "interaction_count",
    "complaint_count",
    "refund_request_count",
    "resolved_issue_rate",
    "avg_resolution_hours",
    "avg_satisfaction_score",
    "days_since_last_interaction",
    "complaint_rate",
    "refund_request_rate",
}

## 2. Load Source Data

In [3]:
customers = pd.read_csv(
    RAW_DATA_DIR / "customers.csv",
    parse_dates=["signup_date"],
)

activities = pd.read_csv(
    RAW_DATA_DIR / "activities.csv",
    parse_dates=["activity_date"],
)

transactions = pd.read_csv(
    RAW_DATA_DIR / "transactions.csv",
    parse_dates=["transaction_date"],
)

interactions = pd.read_csv(
    RAW_DATA_DIR / "interactions.csv",
    parse_dates=["interaction_date"],
)

modeling_labels = pd.read_csv(
    PROCESSED_DATA_DIR / "modeling_labels.csv",
    parse_dates=["snapshot_date"],
)

In [4]:
print(
    {
        "customers": customers.shape,
        "activities": activities.shape,
        "transactions": transactions.shape,
        "interactions": interactions.shape,
        "modeling_labels": modeling_labels.shape,
    }
)

{'customers': (5000, 9), 'activities': (900000, 8), 'transactions': (5513, 14), 'interactions': (8674, 9), 'modeling_labels': (34905, 3)}


## 3. Build Feature Domains

In [5]:
feature_base = (
    modeling_labels
    .merge(
        customers,
        on="customer_id",
        how="left",
        validate="many_to_one",
    )
)

feature_base["tenure_days"] = (
    feature_base["snapshot_date"]
    - feature_base["signup_date"]
).dt.days

In [6]:
activity_features = build_activity_features(
    labels=modeling_labels,
    activities=activities,
    windows=ACTIVITY_WINDOWS,
)

In [7]:
transaction_features = build_transaction_features(
    labels=modeling_labels,
    transactions=transactions,
)

In [8]:
interaction_features = build_interaction_features(
    labels=modeling_labels,
    interactions=interactions,
)

In [9]:
print(
    {
        "profile": feature_base.shape,
        "activity": activity_features.shape,
        "transactions": transaction_features.shape,
        "interactions": interaction_features.shape,
    }
)

{'profile': (34905, 12), 'activity': (34905, 46), 'transactions': (34905, 16), 'interactions': (34905, 12)}


## 4. Assemble Feature Matrix

In [10]:
feature_base = (
    feature_base
    .merge(
        activity_features,
        on=SNAPSHOT_KEY,
        how="left",
        validate="one_to_one",
    )
    .merge(
        transaction_features,
        on=SNAPSHOT_KEY,
        how="left",
        validate="one_to_one",
    )
    .merge(
        interaction_features,
        on=SNAPSHOT_KEY,
        how="left",
        validate="one_to_one",
    )
)

In [11]:
modeling_dataset = feature_base.drop(
    columns=[
        "signup_date",
    ],
    errors="ignore",
).copy()

In [12]:
print("Final modeling dataset:", modeling_dataset.shape)

Final modeling dataset: (34905, 79)


## 5. Feature Matrix Data Contract

In [13]:
validate_feature_matrix(
    modeling_dataset,
    snapshot_key=SNAPSHOT_KEY,
    target_col=TARGET_COLUMN,
    forbidden_features=FORBIDDEN_FEATURES,
    expected_nullable=EXPECTED_NULLABLE_FEATURES,
)

print("Feature matrix validation passed.")

Feature matrix validation passed.


In [14]:
print(
    {
        "rows": len(modeling_dataset),
        "features": modeling_dataset.shape[1] - 3,
        "snapshots": modeling_dataset["snapshot_date"].nunique(),
        "customers": modeling_dataset["customer_id"].nunique(),
        "churn_rate": round(modeling_dataset["churn"].mean(), 3),
    }
)

{'rows': 34905, 'features': 76, 'snapshots': 7, 'customers': 4998, 'churn_rate': np.float64(0.201)}


## 6. Persist Feature Dataset

The validated customer-snapshot feature matrix is persisted as the canonical input for downstream churn modeling.

In [15]:
output_path = (
    PROCESSED_DATA_DIR
    / "modeling_features.parquet"
)

modeling_dataset.to_parquet(
    output_path,
    index=False,
)

print(f"Saved feature matrix to: {output_path}")

Saved feature matrix to: /Users/qiaodou/Desktop/onemoreclass-churn-intelligence/data/processed/modeling_features.parquet
